# ASR Russian Numbers — Training in Google Colab

**Before running:** Runtime → Change runtime type → **T4 GPU**

Steps:
1. Mount Google Drive (for saving checkpoints)
2. Clone your GitHub repo
3. Install dependencies
4. Upload competition data
5. Train the model
6. Download best checkpoint

In [ ]:
# ── 0. Configuration — UPDATE before running ─────────────────────────────────
GITHUB_REPO = "YOUR_USERNAME/asr-russian-numbers"  # e.g. "alexcodeaap/asr-russian-numbers"

# Where to save checkpoints on your Google Drive
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/asr-russian-numbers/checkpoints"

# Where the competition data will be placed
DATA_DIR = "/content/data"

In [ ]:
# ── 1. Check GPU ──────────────────────────────────────────────────────────────
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("WARNING: No GPU! Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# ── 2. Mount Google Drive (to save checkpoints between sessions) ──────────────
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoints will be saved to: {DRIVE_CHECKPOINT_DIR}")

In [ ]:
# ── 3. Clone GitHub repo ──────────────────────────────────────────────────────
import os

REPO_DIR = "/content/asr-russian-numbers"

if os.path.exists(REPO_DIR):
    !rm -rf {REPO_DIR}

!git clone https://github.com/{GITHUB_REPO}.git {REPO_DIR}

%cd {REPO_DIR}
print("Working directory:", os.getcwd())

In [ ]:
# ── 4. Install dependencies ───────────────────────────────────────────────────
!pip install -q num2words editdistance torchaudio torchmetrics soundfile
print("Done.")

In [ ]:
# ── 5. Upload competition data ────────────────────────────────────────────────
# Option A: Upload via Kaggle API (recommended)
# Replace KAGGLE_COMPETITION with the actual competition slug

KAGGLE_COMPETITION = "YOUR_COMPETITION_SLUG"  # e.g. "itmo-speech-russian-numbers"

# Upload your kaggle.json (from kaggle.com → Account → API → Create New Token)
from google.colab import files
print("Upload your kaggle.json file:")
uploaded = files.upload()  # select kaggle.json from your computer

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!pip install -q kaggle
!kaggle competitions download -c {KAGGLE_COMPETITION} -p {DATA_DIR}
!cd {DATA_DIR} && unzip -q *.zip

print("Data downloaded to:", DATA_DIR)
!ls {DATA_DIR}

In [ ]:
# ── 5b. Alternative: Upload data manually ────────────────────────────────────
# Use this cell INSTEAD of cell 5 if you have the data as a zip file

# from google.colab import files
# import zipfile, os
# print("Upload your data zip file:")
# uploaded = files.upload()
# zip_name = list(uploaded.keys())[0]
# os.makedirs(DATA_DIR, exist_ok=True)
# with zipfile.ZipFile(zip_name, 'r') as z:
#     z.extractall(DATA_DIR)
# print("Extracted to:", DATA_DIR)
# !ls {DATA_DIR}

In [ ]:
# ── 6. Validate data ──────────────────────────────────────────────────────────
!python prepare_data.py --csv {DATA_DIR}/train.csv --audio_dir {DATA_DIR}/ --stats
!python prepare_data.py --csv {DATA_DIR}/dev.csv   --audio_dir {DATA_DIR}/ --stats

In [ ]:
# ── 7. Patch config to point to the data ─────────────────────────────────────
import yaml

with open("config.yaml") as f:
    cfg = yaml.safe_load(f)

cfg["data"]["train_csv"]       = f"{DATA_DIR}/train.csv"
cfg["data"]["dev_csv"]         = f"{DATA_DIR}/dev.csv"
cfg["data"]["train_audio_dir"] = f"{DATA_DIR}/"
cfg["data"]["dev_audio_dir"]   = f"{DATA_DIR}/"
cfg["training"]["checkpoint_dir"] = DRIVE_CHECKPOINT_DIR + "/"

# Reduce workers for Colab stability
cfg["training"]["num_workers"] = 2

with open("config_colab.yaml", "w") as f:
    yaml.dump(cfg, f)

print("config_colab.yaml created")
!cat config_colab.yaml

In [ ]:
# ── 8. Train! ─────────────────────────────────────────────────────────────────
# Checkpoints are saved directly to Google Drive after each epoch.
# If the session disconnects, resume with --resume (see next cell).

!python train.py --config config_colab.yaml

In [ ]:
# ── 8b. Resume training after a disconnect ───────────────────────────────────
# Run cells 1–7 first to restore the environment, then run this cell.

# !python train.py --config config_colab.yaml --resume {DRIVE_CHECKPOINT_DIR}/last.pt

In [ ]:
# ── 9. Evaluate best checkpoint ───────────────────────────────────────────────
!python evaluate.py --config config_colab.yaml --checkpoint {DRIVE_CHECKPOINT_DIR}/best.pt

In [ ]:
# ── 10. Download best.pt to your computer ────────────────────────────────────
# You need to upload best.pt to a GitHub Release manually.
# Option A: it's already in your Google Drive — just download it from Drive.
# Option B: download directly from Colab:

from google.colab import files
files.download(f"{DRIVE_CHECKPOINT_DIR}/best.pt")